# Лабораторная работа № 4
## Вычисления с использованием языка ассемблера (FASM)

**Вариант:** 19  
**Среда выполнения:** FASM, Linux x86 (32-bit)

**Цель работы:** научиться составлять простые программы на языке ассемблера, использовать регистры, арифметические операции, условные переходы и системные вызовы для ввода и вывода данных.

> Примечание: в этом отчёте задачи оформлены как исходный код FASM. Ячейки с ассемблерным кодом предназначены для просмотра и копирования; они не выполняются как Python-код в Jupyter Notebook.

## Задание для варианта

По выбранному варианту выполняются задачи № 1, 9 и 11:

1. Посчитать количество чисел от 1 до `n`, которые делятся одновременно на 37 и 13.
2. Вычислить сумму `1 + 2 - 3 - 4 + 5 + 6 - 7 - 8 + ...` до `n`.
3. По голосам `n` судей определить, принято ли решение большинством голосов «Да» (1) или «Нет» (0).

## Подготовка

Программы написаны для FASM под Linux x86 (32-bit). Для каждой задачи используется ввод через системный вызов `read` (`int 80h`) и вывод через `write`.

Файлы `task1.asm`, `task9.asm`, `task11.asm` и `common.inc` должны находиться в одной папке.

## Общие процедуры: `common.inc`

В файле находятся процедуры ввода целого числа, вывода числа и завершения программы. Этот код подключается к каждой программе директивой `include 'common.inc'`.

```asm
read_int:
    push edi
    push esi

    mov edi, 0
    mov esi, 1

read_loop:
    mov eax, 3
    mov ebx, 0
    mov ecx, char_buf
    mov edx, 1
    int 80h

    cmp eax, 1
    jne read_end

    mov al, [char_buf]

    cmp al, 10
    je read_end

    cmp al, '-'
    jne check_digit

    mov esi, -1
    jmp read_loop

check_digit:
    cmp al, '0'
    jb read_loop

    cmp al, '9'
    ja read_loop

    movzx eax, al
    sub eax, '0'

    imul edi, 10
    add edi, eax

    jmp read_loop

read_end:
    mov eax, edi
    imul eax, esi

    pop esi
    pop edi
    ret


print_int:
    mov esi, out_buf
    add esi, 31

    mov byte [esi], 10
    dec esi

    mov ebx, 10
    mov ebp, 0

    cmp eax, 0
    jge print_loop

    neg eax
    mov ebp, 1

print_loop:
    xor edx, edx
    div ebx

    add dl, '0'
    mov [esi], dl
    dec esi

    test eax, eax
    jnz print_loop

    cmp ebp, 1
    jne print_end

    mov byte [esi], '-'
    dec esi

print_end:
    inc esi

    mov edx, out_buf
    add edx, 31
    sub edx, esi
    inc edx

    mov eax, 4
    mov ebx, 1
    mov ecx, esi
    int 80h
    ret


exit_program:
    mov eax, 1
    mov ebx, 0
    int 80h
```

## Задача № 1. Числа, кратные 37 и 13

**Условие:** для заданного `n` определить количество целых чисел от 1 до `n`, которые делятся одновременно на 37 и 13.

**Алгоритм:** числа 37 и 13 взаимно простые, поэтому искомые числа делятся на `37 × 13 = 481`. Количество таких чисел равно целой части `n / 481`.

**Листинг `task1.asm`:**

```asm
format ELF executable 3
entry start

segment readable executable

start:
    call read_int

    mov ebx, 481
    xor edx, edx
    div ebx

    call print_int
    jmp exit_program

include 'common.inc'

segment readable writeable

char_buf rb 1
out_buf rb 32
```

**Пример проверки**

Ввод: `1000`  
Ожидаемый результат: `2`, поскольку числа 481 и 962 делятся одновременно на 37 и 13.

| `n` | Результат |
|---:|---:|
| 100 | 0 |
| 481 | 1 |
| 1000 | 2 |

## Задача № 9. Вычисление суммы

**Условие:** вычислить сумму

\[
S = 1 + 2 - 3 - 4 + 5 + 6 - 7 - 8 + \cdots
\]

до заданного `n`.

**Алгоритм:** программа перебирает числа от 1 до `n`. Для определения знака используется номер пары: пары `(1, 2)`, `(5, 6)` и т. д. складываются, а пары `(3, 4)`, `(7, 8)` и т. д. вычитаются.

**Листинг `task9.asm`:**

```asm
format ELF executable 3
entry start

segment readable executable

start:
    call read_int

    mov ebp, eax
    mov esi, 1
    mov edi, 0

cycle:
    cmp esi, ebp
    jg finish

    mov eax, esi
    dec eax
    shr eax, 1
    and eax, 1

    cmp eax, 0
    jne minus

    add edi, esi
    jmp next

minus:
    sub edi, esi

next:
    inc esi
    jmp cycle

finish:
    mov eax, edi
    call print_int
    jmp exit_program

include 'common.inc'

segment readable writeable

char_buf rb 1
out_buf rb 32
```

**Пример проверки**

При `n = 6`:

\[
S = 1 + 2 - 3 - 4 + 5 + 6 = 7
\]

Ожидаемый результат: `7`.

| `n` | Сумма |
|---:|---:|
| 1 | 1 |
| 2 | 3 |
| 4 | -4 |
| 6 | 7 |
| 8 | -4 |

## Задача № 11. Голосование судей

**Условие:** каждый из `n` судей голосует «Да» (1) или «Нет» (0). Решение принимается большинством голосов.

**Алгоритм:** программа считывает количество судей, затем по одному считывает каждый голос. Если число голосов «Да» строго больше половины от общего числа судей, выводится `Да`. В противном случае выводится `Нет`.

**Листинг `task11.asm`:**

```asm
format ELF executable 3
entry start

segment readable executable

start:
    call read_int

    mov ebp, eax
    mov esi, 0
    mov edi, 0

vote_loop:
    cmp edi, ebp
    jge result

    call read_int

    cmp eax, 1
    jne next_vote

    inc esi

next_vote:
    inc edi
    jmp vote_loop

result:
    mov eax, ebp
    shr eax, 1

    cmp esi, eax
    jle no

yes:
    mov eax, 4
    mov ebx, 1
    mov ecx, yes_msg
    mov edx, yes_len
    int 80h

    jmp exit_program

no:
    mov eax, 4
    mov ebx, 1
    mov ecx, no_msg
    mov edx, no_len
    int 80h

    jmp exit_program

include 'common.inc'

segment readable writeable

char_buf rb 1
out_buf rb 32

yes_msg db 'Da', 10
yes_len = $ - yes_msg

no_msg db 'Net', 10
no_len = $ - no_msg
```

**Пример проверки**

Ввод:
```text
5
1
1
0
1
0
```

Ожидаемый результат: `Da`, поскольку 3 из 5 судей проголосовали «Да».

В исходном коде сообщения записаны латиницей (`Da`, `Net`), чтобы избежать проблем с кодировкой строк в ассемблерном файле. При необходимости их можно заменить на русские строки в UTF-8.

## Компиляция и запуск

Сохраните код каждой программы в соответствующий файл, а общий код — в `common.inc`. Затем в терминале Linux выполните:

```bash
fasm task1.asm task1
fasm task9.asm task9
fasm task11.asm task11
```

Запуск:

```bash
./task1
./task9
./task11
```

Программы рассчитаны на Linux x86 (32-bit) и используют системные вызовы Linux через `int 80h`. Если система не поддерживает запуск 32-битных исполняемых файлов, может понадобиться установить соответствующие библиотеки или использовать 32-битную виртуальную машину.

## Вывод

В ходе лабораторной работы были рассмотрены основные элементы языка ассемблера FASM: регистры общего назначения, арифметические команды, сравнения, условные и безусловные переходы, циклы и системные вызовы ввода-вывода.

Были составлены программы для подсчёта чисел, кратных двум заданным значениям, вычисления суммы с чередованием знаков и определения результата голосования большинства.